# Original baseline (v1)

This is the first version of the project, kept for reference and as the `baseline` configuration in `eval/run_eval.py`. Tutorial leftovers and the OpenAI code path have been removed.

Approach: `PyPDFLoader.load_and_split()` (default recursive splitter, ~4000 chars) → in-memory vector store → top-4 similarity search → single prompt to Llama 3.1.

Extra dependencies for this notebook only: `pip install langchain-community pypdf docarray`.

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.output_parsers import StrOutputParser

model = ChatOllama(model="llama3.1:8b")
embeddings = OllamaEmbeddings(model="nomic-embed-text")
parser = StrOutputParser()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../data/raw/thesis.pdf")
pages = loader.load_and_split()
len(pages)

In [ ]:

from langchain_core.prompts import PromptTemplate
template = """
Answer the question based on the following text. 
If you can't answer the question, please write "I don't know."

context={context}


Question: {question}
"""
prompt = PromptTemplate.from_template(template)


In [ ]:
from langchain_community.vectorstores import DocArrayInMemorySearch

vectorstore = DocArrayInMemorySearch.from_documents(
    pages, 
    embedding=embeddings
)

In [ ]:
retriever = vectorstore.as_retriever()
retriever.invoke("Quantitative Trait Loci (QTL) mapping")

In [ ]:
from operator import itemgetter

chain = (
    {
        "context":itemgetter("question") | retriever,
        "question": itemgetter("question") 
    }
    | prompt 
    | model 
    | parser 
)
chain.invoke({"question": "What is the main idea of this text?"})